CODE FOR PARTITIONED CONVOLUTION   

Algorithm: Uniform Partitioned Convloution 

for ease all are in power of 2 

-> Seed lenght = N 
-> Seed block length = B 
-> number of blocks = N/B  

-> input week-key length = M 
-> block Length = B 
-> number of blocks = M/B 

Y[i,j] = X[i]*S[j]

How to combine the results: 
shift the Y[i,j] block to (i+j) in the final output 

In [ ]:
import numpy as np
import math

N = 2**10
M = 2**10
B = 2**5

seed = np.random.randint(1, 10, N)
wkey = np.random.randint(1, 10, M)

pad_N = math.ceil(N / B) * B
pad_M = math.ceil(M / B) * B

seed_pad = np.pad(seed, (0, pad_N - N))
wkey_pad = np.pad(wkey, (0, pad_M - M))

num_seed_blocks = pad_N // B
num_w_blocks = pad_M // B

seed_blocks = [seed_pad[i*B:(i+1)*B] for i in range(num_seed_blocks)]
wkey_blocks = [wkey_pad[i*B:(i+1)*B] for i in range(num_w_blocks)]

z = np.zeros(pad_N + pad_M - 1)

for i in range(num_seed_blocks):
    for j in range(num_w_blocks):

        conv = np.convolve(wkey_blocks[j], seed_blocks[i])

        shift = (i + j) * B

        z[shift:shift + len(conv)] += conv

z = z[:N + M - 1]

z_ref = np.convolve(wkey, seed)

print("Partitioned:", z)
print("Reference  :", z_ref)
print("Match      :", np.array_equal(z, z_ref))

Partitioned: [42. 68. 98. ... 94. 73. 35.]
Reference  : [42 68 98 ... 94 73 35]
Match      : True


In [1]:
## stage one buffer filler 2 - 2
# stage 2 buffer filler 4 - 4
# stage 3 buffer filler 8 - 8 
import numpy as np
N = 1024
NTTstages= int(np.log2(N))

buffer_lengths = (np.zeros(NTTstages))

for stage in range(1,NTTstages):
    buffer_1 = 0
    buffer_2 = 0
    count    = 0
    state    = 0
    reads    = 0
    max_buffer_1 = buffer_1
    max_buffer_2 = buffer_2

    while(count < N):
        buffer_1 += 2**stage - reads
        buffer_2 += 2**stage - reads
        count    += 2**(stage + 1)
        
        if (state == 2):
            reads = 2**(stage) 

        elif (state == 1 and stage < 2):
            state = 2

        elif (state == 1 and stage >= 2):
            reads = 2**(stage)
            state     = 2
    
        # logic for reduction of stage
        elif (state == 0 and stage <= 2):
            state = 1

        elif (state == 0 and stage > 2):
            reads = 2**(stage - 1) - 2
            state = 1
        print(f"Stage: {stage}, reads: {reads} Count: {count}, State: {state}")
        max_buffer_1 = max(max_buffer_1, buffer_1)

    buffer_lengths[stage-1] = max_buffer_1

    # update maximum values
    

print("Buffer Lengths", buffer_lengths)

Stage: 1, reads: 0 Count: 4, State: 1
Stage: 1, reads: 0 Count: 8, State: 2
Stage: 1, reads: 2 Count: 12, State: 2
Stage: 1, reads: 2 Count: 16, State: 2
Stage: 1, reads: 2 Count: 20, State: 2
Stage: 1, reads: 2 Count: 24, State: 2
Stage: 1, reads: 2 Count: 28, State: 2
Stage: 1, reads: 2 Count: 32, State: 2
Stage: 1, reads: 2 Count: 36, State: 2
Stage: 1, reads: 2 Count: 40, State: 2
Stage: 1, reads: 2 Count: 44, State: 2
Stage: 1, reads: 2 Count: 48, State: 2
Stage: 1, reads: 2 Count: 52, State: 2
Stage: 1, reads: 2 Count: 56, State: 2
Stage: 1, reads: 2 Count: 60, State: 2
Stage: 1, reads: 2 Count: 64, State: 2
Stage: 1, reads: 2 Count: 68, State: 2
Stage: 1, reads: 2 Count: 72, State: 2
Stage: 1, reads: 2 Count: 76, State: 2
Stage: 1, reads: 2 Count: 80, State: 2
Stage: 1, reads: 2 Count: 84, State: 2
Stage: 1, reads: 2 Count: 88, State: 2
Stage: 1, reads: 2 Count: 92, State: 2
Stage: 1, reads: 2 Count: 96, State: 2
Stage: 1, reads: 2 Count: 100, State: 2
Stage: 1, reads: 2 Count: 

Number theoretic transforms: theoretical output

In [ ]:
import numpy as np 

#modulus for montgomery reduction 
mod_mlkem = 3329
mod_mldsa = 8380417
#auxililary variable for montgomery reduction
R = 2**32  
# Inverse of auxililary variable 'R'
R_inv_mlkem = pow(R, -1, mod_mlkem)
R_inv_mldsa = pow(R, -1, mod_mldsa)

# montgomery convertor
def montgomery_convertor(x, q, R):
    return (x * R) % q

# montgomery reduction
def montgomery_reduction(x, q, R, R_inv):
    return (x * R_inv) % q